In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup 

In [ ]:
url = "https://www.ccny.cuny.edu/registrar/fall"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()
print(f"Status Code: {response.status_code}")

In [ ]:
soup = BeautifulSoup(response.content, "html.parser")
tables = soup.find_all("table")

raw_dfs = pd.read_html(str(tables))
df = raw_dfs[0]

df = df.iloc[:, :2]
df.columns = ["date_raw", "text"]
df.head()

In [ ]:
YEAR = 2021

# Clean range strings (e.g., "August 25 - August 27" -> "August 25")
df["clean_date_str"] = df["date_raw"].astype(str).str.split("-").str[0].str.strip()

# Convert to pandas datetime
df["date"] = pd.to_datetime(
    df["clean_date_str"] + f" {YEAR}", 
    format="%B %d %Y", 
    errors="coerce"
)

# Filter out invalid header rows
df = df.dropna(subset=["date"]).copy()

# Add day of week column ('dow')
df["dow"] = df["date"].dt.day_name()

# Set index to standard Python date objects
df["date"] = df["date"].dt.date
df.set_index("date", inplace=True)

# Select final requested columns
df = df[["dow", "text"]]
df.head(10)

In [7]:
#import packages
import pandas as pd
import requests
from bs4 import  BeautifulSoup


#specifying url
url = "https://www.ccny.cuny.edu/registrar/fall"

#a standard user-agent header to avoid receiving HTTP 403 Forbidden errors
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

# Package the request, send the request and catch the response: r
r = requests.get(url, headers=headers)

# Extract the response as html: html_doc
html_doc = r.text

# Create BeautifulSoup object using html.parser
soup = BeautifulSoup(html_doc, "html.parser")

json_data = r.json()

# Assuming rows are inside <tr> tags in a table
rows = soup.find_all("tr")

for row in rows:
    cols = row.find_all(["td", "th"])
    cols = [ele.text.strip() for ele in cols]

    # Ensure row has expected number of columns (3 columns: Date, DOW, Text)
    if len(cols) == 3:
        json_data.append(cols)

# Convert to DataFrame
df = pd.DataFrame(json_data[1:], columns=["date", "dow", "text"])
print(df)



JSONDecodeError: Expecting value: line 1 column 1 (char 0)